# Lab 4: Preference-Based Post-Training -- DPO and GRPO

In this lab, you will implement two fundamental post-training algorithms from scratch using the Tinker training API:

1. **DPO (Direct Preference Optimization)**: Learn from human preference data to make a model more helpful and harmless
2. **GRPO (Group Relative Policy Optimization)**: Use verifiable rewards to improve a model's math reasoning

You will:
- Set up Tinker and understand its training primitives
- Fine-tune an SFT baseline model
- Implement the DPO loss function and train on preference pairs
- Implement GRPO with verifiable math rewards
- Compare all methods

---

## Section 0: Tinker Setup & Basics

Tinker is a cloud-based training API that lets you fine-tune large language models without needing a local GPU. In this section, you will install and configure Tinker, then verify that training and inference work correctly.

Nothing to implement here -- just run each cell and check the outputs.

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
SEED = 42
NUM_SFT_SAMPLES = 300
NUM_DPO_PAIRS = 300
NUM_GRPO_PROBLEMS = 200
DPO_BETA = 0.1
GRPO_GROUP_SIZE = 4
SFT_EPOCHS = 1
DPO_EPOCHS = 1
GRPO_ITERATIONS = 50
BATCH_SIZE = 4
LEARNING_RATE = 2e-4

print("Configuration loaded.")

In [ ]:
# =============================================================================
# Imports
# =============================================================================

import torch
import json
import random
import math
import re
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tinker
from tinker import types
from transformers import AutoTokenizer
from datasets import load_dataset
import warnings

warnings.filterwarnings("ignore")

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Visualization defaults
plt.style.use('seaborn-v0_8-whitegrid')
COLOR_SFT = '#4C72B0'       # blue -- SFT baseline
COLOR_DPO = '#DD8452'        # orange -- DPO model
COLOR_GRPO = '#55A868'       # green -- GRPO model

print("All imports loaded successfully.")

In [ ]:
# =============================================================================
# Connect to Tinker
# =============================================================================

service = tinker.ServiceClient()
capabilities = service.get_server_capabilities()

print("=" * 60)
print("Connected to Tinker!")
print("=" * 60)
print(f"\nSupported models:")
for model in capabilities.supported_models:
    print(f"  - {model.name}")
print()
print("Connection verified successfully.")

In [ ]:
# =============================================================================
# Load Tokenizer and Test Sampling
# =============================================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Create a sampling client for the base model
base_sampling_client = service.create_sampling_client(model_path=MODEL_NAME)

# Test with a simple prompt
test_messages = [{"role": "user", "content": "What is 2 + 2?"}]
test_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
test_tokens = tokenizer.encode(test_text)

test_input = types.ModelInput.from_ints(tokens=test_tokens)
test_params = types.SamplingParams(max_tokens=64, temperature=0.1)

test_result = base_sampling_client.sample(
    prompt=test_input, sampling_params=test_params, num_samples=1
).result()

print("Base model test response:")
print(tokenizer.decode(test_result.sequences[0].tokens, skip_special_tokens=True))
print("\nSampling client works!")

In [ ]:
# =============================================================================
# Test Training Client
# =============================================================================

# Create a LoRA training client
test_training_client = service.create_lora_training_client(
    base_model=MODEL_NAME, rank=16
)

# Test a single forward pass
test_datum = types.Datum(
    model_input=types.ModelInput.from_ints(tokens=test_tokens[:50]),
    loss_fn_inputs=dict(
        target_tokens=test_tokens[1:51],
        weights=[1.0] * 50,
    )
)

test_fwd = test_training_client.forward([test_datum], "cross_entropy").result()
print(f"Forward pass test -- loss: {test_fwd.metrics.get('loss', 'N/A')}")
print(f"Logprobs shape: {len(test_fwd.loss_fn_outputs[0]['logprobs'].data)} values")
print("\nTraining client works! Tinker is fully set up.")

# Clean up test client
del test_training_client

---

## Section 1: SFT Baseline

Before we can do preference-based training, we need a supervised fine-tuned (SFT) model as our starting point. SFT trains the model with standard next-token prediction on high-quality demonstration data.

We will use a subset of the UltraFeedback dataset, taking only the "chosen" (preferred) responses as demonstrations.

In [ ]:
# =============================================================================
# Load SFT Training Data
# =============================================================================

# Load UltraFeedback dataset -- we use the "chosen" responses for SFT
raw_dataset = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_sft")

# Take a subset and format as conversations
sft_data = []
for i, sample in enumerate(raw_dataset):
    if i >= NUM_SFT_SAMPLES:
        break
    messages = sample["chosen"]
    # Only keep samples with user + assistant messages
    if len(messages) >= 2 and messages[0]["role"] == "user" and messages[1]["role"] == "assistant":
        sft_data.append({
            "prompt": messages[0]["content"],
            "response": messages[1]["content"],
        })

print(f"Loaded {len(sft_data)} SFT training samples.")
print(f"\nExample prompt (first 200 chars):")
print(f"  {sft_data[0]['prompt'][:200]}...")
print(f"\nExample response (first 200 chars):")
print(f"  {sft_data[0]['response'][:200]}...")

In [ ]:
# =============================================================================
# PROVIDED: Prepare SFT Training Datums
# =============================================================================

def prepare_sft_datum(sample, tokenizer, max_length=512):
    """Convert a training sample into a Tinker Datum for SFT training.
    
    Tokenizes the prompt + response as a chat conversation, then creates
    a Datum with target tokens shifted by one position (next-token prediction).
    Only the assistant's response tokens are weighted for the loss.
    
    Args:
        sample: Dict with 'prompt' and 'response' keys
        tokenizer: HuggingFace tokenizer
        max_length: Maximum sequence length
    
    Returns:
        types.Datum ready for forward_backward()
    """
    messages = [
        {"role": "user", "content": sample["prompt"]},
        {"role": "assistant", "content": sample["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tokens = tokenizer.encode(text, truncation=True, max_length=max_length)
    
    # Create target tokens (shifted by 1)
    target_tokens = tokens[1:] + [tokenizer.pad_token_id]
    
    # Create weights -- only train on assistant response tokens
    # Find where the assistant response starts
    prompt_messages = [{"role": "user", "content": sample["prompt"]}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    prompt_len = len(tokenizer.encode(prompt_text, truncation=True, max_length=max_length))
    
    weights = [0.0] * prompt_len + [1.0] * (len(tokens) - prompt_len)
    weights = weights[:len(tokens)]  # Ensure same length
    
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=tokens),
        loss_fn_inputs=dict(target_tokens=target_tokens, weights=weights),
    )


# Prepare all SFT datums
sft_datums = [prepare_sft_datum(s, tokenizer) for s in tqdm(sft_data, desc="Preparing SFT data")]
print(f"\nPrepared {len(sft_datums)} SFT datums.")
print(f"Example datum token length: {sft_datums[0].model_input.length}")

### 1.1 Train the SFT Model

**TODO**: Write the SFT training loop. Use `forward_backward()` with `"cross_entropy"` loss and `optim_step()` to train for `SFT_EPOCHS` epochs.

This is standard supervised fine-tuning -- iterate over batches of datums, compute the cross-entropy loss, and update the model weights.

In [ ]:
# TODO: Train the SFT model (~15-20 lines)
# Hint: 
#   1. Create a training client: service.create_lora_training_client(base_model=MODEL_NAME, rank=16)
#   2. Loop over epochs and batches (use BATCH_SIZE)
#   3. For each batch: call training_client.forward_backward(batch, "cross_entropy")
#   4. Then call training_client.optim_step(types.AdamParams(learning_rate=LEARNING_RATE))
#   5. Track the loss from result.metrics for plotting
#   6. Print epoch-level average loss
# Store the training client as `sft_training_client` and losses as `sft_losses`.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# =============================================================================
# Save SFT Model and Test Generation
# =============================================================================

# Get a sampling client from the trained SFT model
sft_sampling_client = sft_training_client.save_weights_and_get_sampling_client()

# Also save a named checkpoint for later reference
sft_training_client.save_state("sft-baseline-lab04").result()

# Test the SFT model
def generate_response(sampling_client, tokenizer, prompt, max_tokens=256, temperature=0.7):
    """Generate a response from a sampling client."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    
    model_input = types.ModelInput.from_ints(tokens=tokens)
    params = types.SamplingParams(
        max_tokens=max_tokens,
        temperature=max(temperature, 0.01),
        stop=[tokenizer.eos_token] if tokenizer.eos_token else [],
    )
    
    result = sampling_client.sample(
        prompt=model_input, sampling_params=params, num_samples=1
    ).result()
    
    return tokenizer.decode(result.sequences[0].tokens, skip_special_tokens=True)


# Test on a few prompts
test_prompts = [
    "Explain the difference between a list and a tuple in Python.",
    "What are some healthy breakfast ideas?",
    "Write a haiku about machine learning.",
]

print("SFT Model Responses:")
print("=" * 60)
for prompt in test_prompts:
    response = generate_response(sft_sampling_client, tokenizer, prompt)
    print(f"\nPrompt: {prompt}")
    print(f"Response: {response[:300]}...")
    print("-" * 60)

---

## Section 2: Direct Preference Optimization (DPO)

DPO is an elegant alternative to RLHF that directly optimizes a policy from preference data, without needing to train a separate reward model.

### The DPO Loss

Given a preference pair (prompt **x**, chosen response **y_w**, rejected response **y_l**), the DPO loss is:

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\left(\beta \left[\log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right]\right)$$

Where:
- $\pi_\theta$ is the model being trained (initialized from SFT)
- $\pi_{\text{ref}}$ is the reference model (the frozen SFT model)
- $\beta$ controls the strength of the KL constraint
- $\sigma$ is the sigmoid function

**Intuition**: DPO increases the probability of chosen responses relative to rejected ones, but penalizes large deviations from the reference model (to prevent the model from degenerating).

In [ ]:
# =============================================================================
# Load Preference Data
# =============================================================================

# Load UltraFeedback preference pairs
pref_dataset = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")

# Take a subset of preference pairs
pref_data = []
for i, sample in enumerate(pref_dataset):
    if i >= NUM_DPO_PAIRS + 50:  # Extra for eval
        break
    chosen = sample["chosen"]
    rejected = sample["rejected"]
    
    # Ensure both have user + assistant format
    if (len(chosen) >= 2 and len(rejected) >= 2 
        and chosen[0]["role"] == "user" and chosen[1]["role"] == "assistant"
        and rejected[0]["role"] == "user" and rejected[1]["role"] == "assistant"):
        pref_data.append({
            "prompt": chosen[0]["content"],
            "chosen": chosen[1]["content"],
            "rejected": rejected[1]["content"],
        })

# Split into train and eval
dpo_train = pref_data[:NUM_DPO_PAIRS]
dpo_eval = pref_data[NUM_DPO_PAIRS:]

print(f"DPO training pairs: {len(dpo_train)}")
print(f"DPO eval pairs:     {len(dpo_eval)}")
print(f"\nExample preference pair:")
print(f"  Prompt:   {dpo_train[0]['prompt'][:150]}...")
print(f"  Chosen:   {dpo_train[0]['chosen'][:150]}...")
print(f"  Rejected: {dpo_train[0]['rejected'][:150]}...")

In [ ]:
# =============================================================================
# PROVIDED: Prepare DPO Datums
# =============================================================================

def prepare_dpo_datum(prompt, response, tokenizer, max_length=512):
    """Create a Datum for a prompt-response pair, suitable for computing log-probs.
    
    Tokenizes prompt + response and creates a Datum where target_tokens are
    shifted by one. Weights are 1.0 only on the response tokens (not the prompt).
    
    Args:
        prompt: The user prompt string
        response: The assistant response string
        tokenizer: HuggingFace tokenizer
        max_length: Maximum sequence length
    
    Returns:
        types.Datum
    """
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tokens = tokenizer.encode(text, truncation=True, max_length=max_length)
    target_tokens = tokens[1:] + [tokenizer.pad_token_id]
    
    # Compute prompt length to mask prompt tokens
    prompt_messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    prompt_len = len(tokenizer.encode(prompt_text, truncation=True, max_length=max_length))
    
    weights = [0.0] * prompt_len + [1.0] * (len(tokens) - prompt_len)
    weights = weights[:len(tokens)]
    
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=tokens),
        loss_fn_inputs=dict(target_tokens=target_tokens, weights=weights),
    )


print("DPO datum helper defined.")

### 2.1 Compute Reference Log-Probabilities

Before training, we need to compute the log-probabilities of both chosen and rejected responses under the **reference model** (our frozen SFT model). These reference log-probs stay fixed throughout DPO training.

**TODO**: Use the SFT training client's `forward()` method to compute per-token log-probs for each chosen and rejected response. Sum the response-token log-probs (using the weights mask) to get the total log-probability of each response.

In [ ]:
# TODO: Compute reference log-probabilities for all DPO pairs (~20-25 lines)
# Hint:
#   1. For each pair in dpo_train, create a Datum for the chosen response
#      and a Datum for the rejected response using prepare_dpo_datum()
#   2. Call sft_training_client.forward([datum], "cross_entropy") to get logprobs
#   3. The result has result.loss_fn_outputs[0]["logprobs"].data -- a flat list of
#      per-token logprobs. Convert to a tensor.
#   4. Multiply by the weights (from datum.loss_fn_inputs) to mask out prompt tokens,
#      then sum to get total log-prob of the response.
#   5. Store these in two lists: ref_logprobs_chosen and ref_logprobs_rejected
#
# Note: This loop will make 2 * NUM_DPO_PAIRS forward passes. Use tqdm for progress.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 2.2 Implement the DPO Loss Function

Now implement the core DPO loss. You will write a custom loss function that Tinker's `forward_backward_custom()` will call. This function receives the current model's per-token logprobs and must return a scalar loss.

The function signature is:
```python
def dpo_loss_fn(data, logprobs_list):
    # data: List[Datum] -- the datums you passed in (2 per pair: chosen + rejected)
    # logprobs_list: List[torch.Tensor] -- per-token logprobs from current model
    #   Each tensor has shape (seq_len,) and has requires_grad=True
    # Returns: (loss, metrics_dict)
```

**Key steps in your loss function:**
1. For each pair, extract the chosen and rejected logprobs tensors
2. Compute weighted sum of logprobs (multiply by weights, then sum) for chosen and rejected
3. Subtract the pre-computed reference logprobs to get log-ratios
4. Apply the DPO formula: `loss = -log(sigmoid(beta * (chosen_ratio - rejected_ratio)))`

In [ ]:
# TODO: Implement the DPO loss function (~20-30 lines)
# Hint:
#   def dpo_loss_fn(data, logprobs_list):
#       total_loss = 0.0
#       num_pairs = len(data) // 2  # data has [chosen_0, rejected_0, chosen_1, rejected_1, ...]
#       for i in range(num_pairs):
#           chosen_logprobs = logprobs_list[2 * i]     # torch.Tensor with grad
#           rejected_logprobs = logprobs_list[2 * i + 1]
#           
#           # Get weights from the datum to mask prompt tokens
#           chosen_weights = torch.tensor(data[2*i].loss_fn_inputs["weights"].data)
#           rejected_weights = torch.tensor(data[2*i+1].loss_fn_inputs["weights"].data)
#           
#           # Sum weighted logprobs to get total log-prob of response
#           pi_chosen = (chosen_logprobs * chosen_weights).sum()
#           pi_rejected = (rejected_logprobs * rejected_weights).sum()
#           
#           # Get pre-computed reference logprobs (from the lists you computed above)
#           # You'll need to access the batch_ref_chosen and batch_ref_rejected
#           # variables from the outer scope (or pass them via closure)
#           ref_chosen = ...   # float, from ref_logprobs_chosen
#           ref_rejected = ... # float, from ref_logprobs_rejected
#           
#           # Compute DPO loss for this pair
#           chosen_ratio = pi_chosen - ref_chosen
#           rejected_ratio = pi_rejected - ref_rejected
#           loss_i = -torch.nn.functional.logsigmoid(DPO_BETA * (chosen_ratio - rejected_ratio))
#           total_loss = total_loss + loss_i
#       
#       loss = total_loss / num_pairs
#       return loss, {"dpo_loss": loss.item()}

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 2.3 DPO Training Loop

Now put it all together. Create a new training client (initialized from the SFT weights), and train with your DPO loss function.

For each batch of preference pairs:
1. Prepare chosen and rejected datums (interleaved: chosen_0, rejected_0, chosen_1, rejected_1, ...)
2. Call `forward_backward_custom()` with your DPO loss function
3. Call `optim_step()` to update the model weights

In [ ]:
# TODO: DPO training loop (~25-30 lines)
# Hint:
#   1. Create a new training client: service.create_lora_training_client(base_model=MODEL_NAME, rank=16)
#   2. Load the SFT weights: dpo_training_client.load_state("sft-baseline-lab04").result()
#   3. Loop over epochs (DPO_EPOCHS) and batches
#   4. For each batch of pairs:
#      a. Prepare datums: [chosen_datum_0, rejected_datum_0, chosen_datum_1, ...]
#      b. Create a closure that captures the correct ref logprobs for this batch
#      c. Call dpo_training_client.forward_backward_custom(batch_datums, loss_fn)
#      d. Call dpo_training_client.optim_step(types.AdamParams(learning_rate=LEARNING_RATE))
#   5. Track losses for plotting
# Store as `dpo_training_client` and `dpo_losses`.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

In [ ]:
# =============================================================================
# Save DPO Model and Compare with SFT
# =============================================================================

dpo_sampling_client = dpo_training_client.save_weights_and_get_sampling_client()

# Compare SFT vs DPO on the same prompts
comparison_prompts = [
    "How do I pick a lock?",
    "Write a persuasive essay about why homework should be banned.",
    "Explain quantum computing to a 5-year-old.",
    "What should I do if I'm feeling really stressed?",
]

print("SFT vs DPO Comparison:")
print("=" * 60)
for prompt in comparison_prompts:
    sft_response = generate_response(sft_sampling_client, tokenizer, prompt, max_tokens=200)
    dpo_response = generate_response(dpo_sampling_client, tokenizer, prompt, max_tokens=200)
    print(f"\nPrompt: {prompt}")
    print(f"\n  SFT: {sft_response[:250]}")
    print(f"\n  DPO: {dpo_response[:250]}")
    print("-" * 60)

### 2.4 Evaluate DPO

**TODO**: Evaluate the DPO model on the held-out eval pairs. For each pair, generate responses from both SFT and DPO models, then compute a simple preference metric: for how many prompts does the DPO response more closely match the "chosen" response (using length similarity and keyword overlap as proxies)?

In [ ]:
# TODO: Evaluate DPO vs SFT on held-out preference pairs (~15-20 lines)
# Hint:
#   1. For each pair in dpo_eval (or a subset of 20):
#      a. Generate a response from sft_sampling_client
#      b. Generate a response from dpo_sampling_client
#      c. Compare both against the "chosen" response using a simple metric:
#         - Compute word overlap: len(set(response.split()) & set(chosen.split()))
#         - Or use response length similarity
#   2. Count how often DPO "wins" (higher overlap with chosen)
#   3. Print the win rate
#   4. Create a bar chart comparing SFT vs DPO win rates
# Use figsize=(10, 6) and COLOR_SFT / COLOR_DPO for the bars.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 3: GRPO -- RL with Verifiable Rewards on Math

GRPO (Group Relative Policy Optimization) is a reinforcement learning method that does not require a learned reward model. Instead, it:

1. **Samples a group** of G responses for each prompt
2. **Scores each response** with a reward function (for math: is the answer correct?)
3. **Normalizes advantages** within the group: $A_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r})}$
4. **Updates the policy** using a weighted log-probability objective

This is particularly powerful for tasks with **verifiable rewards** (RLVR) -- tasks where we can automatically check if an answer is correct, like math problems.

### The GRPO Objective

For a prompt **x** and group of sampled responses $\{y_1, ..., y_G\}$ with rewards $\{r_1, ..., r_G\}$:

$$\mathcal{L}_{\text{GRPO}} = -\frac{1}{G}\sum_{i=1}^{G} A_i \cdot \log \pi_\theta(y_i | x)$$

where $A_i = \frac{r_i - \bar{r}}{\sigma_r + \epsilon}$ is the group-normalized advantage.

In [ ]:
# =============================================================================
# Load Math Data (GSM8K)
# =============================================================================

gsm_dataset = load_dataset("openai/gsm8k", "main", split="train")

# Take a subset
math_data = []
for i, sample in enumerate(gsm_dataset):
    if i >= NUM_GRPO_PROBLEMS + 50:  # Extra for eval
        break
    # Extract the final numerical answer from GSM8K format
    # GSM8K answers end with "#### <number>"
    answer_text = sample["answer"]
    match = re.search(r"####\s*(.+)", answer_text)
    if match:
        final_answer = match.group(1).strip().replace(",", "")
        math_data.append({
            "question": sample["question"],
            "answer": final_answer,
            "full_solution": answer_text,
        })

math_train = math_data[:NUM_GRPO_PROBLEMS]
math_eval = math_data[NUM_GRPO_PROBLEMS:]

print(f"GRPO training problems: {len(math_train)}")
print(f"GRPO eval problems:     {len(math_eval)}")
print(f"\nExample problem:")
print(f"  Question: {math_train[0]['question'][:200]}")
print(f"  Answer:   {math_train[0]['answer']}")

In [ ]:
# =============================================================================
# PROVIDED: Answer Extraction Helper
# =============================================================================

def extract_numerical_answer(text):
    """Extract the final numerical answer from a model's response.
    
    Looks for patterns like:
    - "the answer is 42"
    - "#### 42"
    - "= 42"
    - A number at the very end of the response
    
    Returns the extracted number as a string, or None if no number found.
    """
    # Try "#### <number>" format first (GSM8K style)
    match = re.search(r"####\s*([\d,]+\.?\d*)", text)
    if match:
        return match.group(1).replace(",", "")
    
    # Try "the answer is <number>"
    match = re.search(r"(?:the answer is|answer:|answer is)\s*([\d,]+\.?\d*)", text, re.IGNORECASE)
    if match:
        return match.group(1).replace(",", "")
    
    # Try "= <number>" at end
    match = re.search(r"=\s*([\d,]+\.?\d*)\s*$", text)
    if match:
        return match.group(1).replace(",", "")
    
    # Try last number in the text
    numbers = re.findall(r"[\d,]+\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "")
    
    return None


# Test
print("Answer extraction test:")
print(f"  'The answer is 42'  -> {extract_numerical_answer('The answer is 42')}")
print(f"  '#### 1,234'        -> {extract_numerical_answer('#### 1,234')}")
print(f"  'So 5 + 3 = 8'      -> {extract_numerical_answer('So 5 + 3 = 8')}")

### 3.1 Implement the Reward Function

**TODO**: Write a function that computes a binary reward: 1.0 if the model's extracted answer matches the ground truth, 0.0 otherwise.

In [ ]:
# TODO: Implement the reward function (~8-12 lines)
# Hint:
#   def compute_reward(generated_text, correct_answer):
#       1. Use extract_numerical_answer() to get the model's answer
#       2. If extraction fails, return 0.0
#       3. Compare the extracted answer with correct_answer
#          - Try exact string match first
#          - Then try numeric comparison (float(extracted) == float(correct))
#          - Handle potential ValueError from float conversion
#       4. Return 1.0 if match, 0.0 otherwise
#
# Test your function on a few examples after implementing.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 3.2 Group Sampling and Advantage Computation

**TODO**: For each math problem, sample G responses from the current policy, compute rewards, and calculate group-normalized advantages.

The advantages tell the model which responses in the group were better than average (positive advantage) and which were worse (negative advantage).

In [ ]:
# TODO: Implement group sampling and advantage computation (~20-25 lines)
# Hint:
#   def sample_group_and_compute_advantages(sampling_client, tokenizer, problem, G=GRPO_GROUP_SIZE):
#       """Sample G responses and compute advantages for one problem.
#       
#       Returns:
#           list of dicts, each with keys:
#             'question', 'response', 'reward', 'advantage', 'tokens'
#       """
#       1. Format the prompt: "Solve this math problem step by step. 
#          At the end, write your final answer after '####'.\n\n{question}"
#       2. Tokenize with chat template and add_generation_prompt=True
#       3. Sample G responses using sampling_client.sample(..., num_samples=G)
#          with temperature=0.7, max_tokens=512
#       4. Decode each response and compute its reward
#       5. Compute advantages:
#          - rewards = [r1, r2, ..., rG]
#          - mean_r = mean(rewards)
#          - std_r = std(rewards) + 1e-8
#          - advantages = [(r - mean_r) / std_r for r in rewards]
#       6. Return the list of dicts
#
# Test on one problem after implementing.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 3.3 GRPO Training Loop

**TODO**: Implement the main GRPO training loop. For each iteration:
1. Sample a batch of problems
2. For each problem, sample a group of responses and compute advantages
3. Create datums from the sampled responses with advantages as weights
4. Use `forward_backward()` with the `"importance_sampling"` loss (which accepts per-token weights that encode the advantages)
5. Update model weights with `optim_step()`

The key insight: by setting the per-token weights to the advantages, the importance_sampling loss naturally implements the GRPO policy gradient -- responses with positive advantages get reinforced, while responses with negative advantages get suppressed.

In [ ]:
# TODO: Implement the GRPO training loop (~30-40 lines)
# Hint:
#   1. Create a new training client and load SFT weights:
#      grpo_training_client = service.create_lora_training_client(base_model=MODEL_NAME, rank=16)
#      grpo_training_client.load_state("sft-baseline-lab04").result()
#   2. Get an initial sampling client:
#      grpo_sampling_client = grpo_training_client.save_weights_and_get_sampling_client()
#   3. For each iteration (GRPO_ITERATIONS):
#      a. Sample a random batch of problems from math_train
#      b. For each problem, call sample_group_and_compute_advantages()
#      c. For each response with non-zero advantage:
#         - Tokenize the full conversation (prompt + response)
#         - Create a Datum with the advantage as the weight on response tokens
#         - Specifically: weights = [0.0]*prompt_len + [advantage]*response_len
#      d. Call grpo_training_client.forward_backward(datums, "importance_sampling")
#      e. Call grpo_training_client.optim_step(...)
#      f. Every 10 iterations, update the sampling client:
#         grpo_sampling_client = grpo_training_client.save_weights_and_get_sampling_client()
#      g. Track mean reward per iteration
#   4. Store as `grpo_training_client`, `grpo_sampling_client`, `grpo_rewards`

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 3.4 Evaluate GRPO Model

**TODO**: Evaluate the GRPO model on held-out math problems. Compare accuracy against the base model and SFT model.

In [ ]:
# TODO: Evaluate GRPO on held-out math problems (~20-25 lines)
# Hint:
#   1. Define an evaluation function:
#      def evaluate_math_model(sampling_client, tokenizer, eval_set, num_samples=None):
#          - For each problem, generate a response (temperature=0.1 for deterministic)
#          - Compute reward (correct or not)
#          - Return accuracy (fraction correct)
#   2. Evaluate: base model, SFT model, and GRPO model
#   3. Print accuracy for each
#   4. Create a bar chart comparing all three
#      Use figsize=(10, 6), COLOR_SFT, COLOR_GRPO, and '#808080' for the base model
# Store results as base_math_acc, sft_math_acc, grpo_math_acc

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 4: Comparison & Analysis

Now let's compare all the models and training methods we have explored.

### 4.1 Training Curves

**TODO**: Plot the training curves for SFT, DPO, and GRPO side by side. This helps visualize how each method converges.

In [ ]:
# TODO: Plot training curves for all three methods (~15-20 lines)
# Hint:
#   - Create a figure with 3 subplots side by side: figsize=(18, 5)
#   - Subplot 1: SFT loss over steps (from sft_losses)
#   - Subplot 2: DPO loss over steps (from dpo_losses)
#   - Subplot 3: GRPO mean reward over iterations (from grpo_rewards)
#   - Use COLOR_SFT, COLOR_DPO, COLOR_GRPO respectively
#   - Add titles, axis labels, grid
#   - plt.tight_layout()

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 4.2 Side-by-Side Model Outputs

**TODO**: Generate responses from all models on the same set of prompts to qualitatively compare their behaviors.

In [ ]:
# TODO: Generate and display side-by-side outputs from all models (~15-20 lines)
# Hint:
#   comparison_prompts = [
#       "How do I make a friend feel better after a bad day?",
#       "What is the square root of 144?",
#       "Should I lie on my resume to get a better job?",
#       "Solve: If a train travels 60 mph for 2.5 hours, how far does it go?",
#   ]
#   For each prompt, generate from: base, SFT, DPO, GRPO models
#   Print them in a formatted comparison
#   Use generate_response() helper

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 4.3 Summary Chart

**TODO**: Create a summary visualization showing what each method is good at.

In [ ]:
# TODO: Create a summary comparison chart (~15-20 lines)
# Hint:
#   Create a grouped bar chart with:
#   - X-axis: Model (Base, SFT, DPO, GRPO)
#   - Y-axis: Performance metric
#   - Groups: "Math Accuracy" and "Helpfulness" (use DPO eval win rate as proxy)
#   - Use figsize=(12, 6)
#   - Colors: '#808080' for base, COLOR_SFT, COLOR_DPO, COLOR_GRPO
#   - Add value labels on bars

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 5: Reflection

Take a few minutes to think about the following questions. There are no "right" answers -- the goal is to build intuition about when and why to use different post-training methods.

### 5.1 DPO vs GRPO

When would you choose DPO over GRPO, and vice versa? Consider:
- What kind of data does each method require?
- What tasks is each method best suited for?
- What are the computational tradeoffs?

*Write your response below:*

*Your answer here...*

### 5.2 The Role of the Reference Model

Both DPO and GRPO use a reference model (the SFT model). Why is this important? What would happen if we did not include the KL constraint against the reference model?

*Write your response below:*

*Your answer here...*

### 5.3 Verifiable vs Non-Verifiable Rewards

GRPO works well for math because we can automatically verify answers. But many important tasks (writing quality, helpfulness, safety) do not have easily verifiable rewards.

- How could you adapt GRPO for tasks without verifiable rewards?
- What are the risks of using a learned reward model instead of verifiable rewards?
- How does this connect to the broader challenge of aligning language models?

*Write your response below:*

*Your answer here...*

### 5.4 Scaling and Practical Considerations

In this lab, you trained on small datasets with a single model. In practice, companies train on millions of preference pairs with much larger models.

- How do you think the effectiveness of DPO vs GRPO changes with scale?
- What practical challenges would you expect when scaling these methods?
- What role does data quality play in preference-based training?

*Write your response below:*

*Your answer here...*

---

## Congratulations!

You have completed the RLHF lab. Here is a summary of what you accomplished:

1. **Set up Tinker** and learned to use its training primitives (forward, backward, optimize, sample)
2. **Trained an SFT baseline** using standard next-token prediction on demonstration data
3. **Implemented DPO from scratch** -- writing the loss function and training loop to learn from human preferences
4. **Implemented GRPO from scratch** -- using verifiable math rewards to improve reasoning without human labels
5. **Compared all methods** and understood when to use each approach

These are the core algorithms behind how modern language models like Claude, ChatGPT, and Gemini are trained after pre-training. Understanding them at this level gives you deep insight into how AI alignment works in practice.